# neural_chess_engine -- Colab training launcher

Thin launcher only (per the project's own convention): clones/finds the repo, installs deps, builds any missing tensor cache, then runs `src/train.py` directly so you see live `tqdm` progress in the cell output. No training logic lives in this notebook.

**Before running:** `Runtime > Change runtime type > GPU` (ideally a decent one -- T4/A100/whatever your Colab tier gives you).

If you'd rather upload the project yourself instead of cloning from GitHub, upload/extract it to `/content/chess_bot` before running the cells below -- the clone cell detects it and uses it as-is.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("!! no GPU -- go to Runtime > Change runtime type > GPU before continuing")

In [ ]:
import os

REPO_DIR = "/content/chess_bot"
REPO_URL = "https://github.com/mericberktas/neural_chess_engine.git"
GIT_BRANCH = "run6-gab-see"  # run8 keeps the 22-channel GAB+SEE encoding, just more data

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !cd {REPO_DIR} && git fetch origin && git checkout {GIT_BRANCH} && git pull origin {GIT_BRANCH}
elif os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists (manually uploaded) -- using it as-is, not cloning.")
else:
    !git clone --branch {GIT_BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
# Colab already ships a CUDA-enabled torch -- skip it like runpod_overnight.sh
# does on rented pods, installing our own could mismatch the preinstalled CUDA build.
!grep -v '^torch$' requirements.txt > /tmp/requirements-no-torch.txt
!pip install --quiet -r /tmp/requirements-no-torch.txt

## Config

`CACHED_MONTHS` already have encoded tensors sitting in run6's Drive cache (same 22-channel encoding, reused as-is -- no need to re-encode). `NEW_MONTHS` are the ones harvested locally but never encoded yet; the next cell builds those on demand. `TEST_MONTH` is pinned to 2025-04 (same as run4-7) so test-top1 stays comparable -- NOT passed to train.py (it doesn't consume a test set itself, only train/val), kept here only for reference/a later eval step.

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/src")

DRIVE_BASE = "/content/drive/MyDrive/chess_bot"
RUN_NAME = "run8"

CACHED_MONTHS = [
    "2026-08", "2026-07", "2026-06", "2026-05", "2026-04", "2026-03", "2026-02", "2026-01",
    "2025-12", "2025-11", "2025-10", "2025-09", "2025-08", "2025-07", "2025-06", "2025-05",
]
NEW_MONTHS = [
    "2025-03", "2025-02", "2025-01", "2024-12", "2024-11", "2024-10", "2024-09", "2024-08",
    "2024-07", "2024-06", "2024-05", "2024-04", "2024-03", "2024-02", "2024-01",
    "2023-12", "2023-11", "2023-10", "2023-09", "2023-08",
]
TEST_MONTH = "2025-04"

BATCH_SIZE = 256
LR = 3e-4
VAL_INTERVAL = 2000 * 256 // BATCH_SIZE
EPOCHS = 15
PATIENCE = 4

In [ ]:
from pathlib import Path

month_source_run = {m: "run6" for m in CACHED_MONTHS}
month_source_run.update({m: RUN_NAME for m in NEW_MONTHS})

TRAIN_DIRS, VAL_DIRS = [], []
for month, source_run in month_source_run.items():
    out_dir = Path(DRIVE_BASE) / "tensors" / source_run / month
    if (out_dir / "train").exists() and (out_dir / "val").exists():
        print(f"{month}: cached tensors found at {out_dir}")
    else:
        pgn_source = Path(DRIVE_BASE) / "filtered_pgn" / f"{month}.pgn"
        print(f"== building {month} (not cached under {source_run}) from {pgn_source} ==")
        !python -u src/build_dataset.py --source {pgn_source} --out-dir {out_dir} --split train --max-games 40000
    TRAIN_DIRS.append(str(out_dir / "train"))
    VAL_DIRS.append(str(out_dir / "val"))

print(f"\n{len(TRAIN_DIRS)} train months, {len(VAL_DIRS)} val months")

In [ ]:
CKPT_DIR = f"{DRIVE_BASE}/checkpoints/{RUN_NAME}"
resume_path = f"{CKPT_DIR}/best.pt"
resume_args = []
if os.path.exists(resume_path):
    print(f"found existing checkpoint, resuming from {resume_path}")
    resume_args = ["--resume-from", resume_path]
else:
    print("no existing checkpoint -- starting fresh")

## Train

Runs in the foreground (not `nohup`'d) so `tqdm`'s epoch progress bar renders live in the cell output. Checkpoints (`best.pt`, TensorBoard logs) are written straight to the mounted Drive path -- no separate sync step needed like on a rented pod.

In [ ]:
train_dir_args = " ".join(TRAIN_DIRS)
val_dir_args = " ".join(VAL_DIRS)
resume_arg_str = " ".join(resume_args)

cmd = (
    f"src/train.py --train-dir {train_dir_args} --val-dir {val_dir_args} "
    f"--out-dir {CKPT_DIR} "
    f"--batch-size {BATCH_SIZE} --lr {LR} --d-model 256 --nhead 8 --num-layers 6 --dim-feedforward 1024 "
    f"--val-interval {VAL_INTERVAL} --epochs {EPOCHS} --patience {PATIENCE} "
    f"{resume_arg_str}"
)
# %run instead of !python: !python pipes a SEPARATE subprocess's output through
# Colab's shell-capture, which doesn't handle tqdm's carriage-return bar
# updates -- the bar just never renders. %run executes train.py in this
# notebook's own kernel process instead, so tqdm writes through Colab's
# normal cell-output stream (the same one every "for i in tqdm(range(n))"
# notebook example relies on) and the bar updates live. Safe here because
# train.py uses num_workers=0 (no DataLoader worker processes / forking) --
# build_dataset.py below keeps using !python because it DOES use
# multiprocessing, and forking this kernel after CUDA is already initialized
# in it would be a real hazard.
get_ipython().run_line_magic("run", cmd)